# Crypto CNN — Stage 3 benchmarks (the CNN is meaningless without these)

**Standalone and read-only.** This notebook reads `crypto_market_data_2018_2026.parquet`, the image
stores in `crypto_image_store/`, and the cached predictions in `crypto_preds/`. It **writes only**
`crypto_benchmark_results.csv`. It never trains, never touches the GPU, and never writes into the
CNN's artifacts — so it is safe to run while `crypto_train.ipynb` is still finishing the I20 configs.
Configs whose predictions are not cached yet are simply skipped and can be re-run later.

## Why this stage exists

A CNN Sharpe on its own is not a result. The pre-registered claim is not "the CNN makes money" —
it is **"the CNN beats what you could have done without it."** Three families of benchmark, all
scored on the **identical test rows and construction** as the CNN (same `eval_base`, same
non-overlapping H-day rebalance grid, same 365 annualization, same locked cost grid):

1. **Naive / directional** — buy-and-hold BTC; equal-weight buy-and-hold of the top-100.
2. **Cross-sectional factors** — momentum (`MOM`), 1-month reversal (`STR`), 1-week reversal
   (`WSTR`), and the Han–Zhou–Zhu trend factor (`TREND`). These reuse the exact signals already in
   the parquet.
3. **A Kalman trend/vol state estimator** — a latent-trend predictor adapted from the pairs
   project's `run_ecm_kalman`. This is the "the CNN must beat a principled state-space model"
   benchmark.

## The one framing that must not be fudged: directional vs market-neutral

The CNN, `MOM`, `STR`, `WSTR`, `TREND` and the Kalman signal are all **long-short decile books
(D10 − D1): market-neutral by construction**, no BTC exposure. Buy-and-hold BTC and equal-weight
top-100 are **long-only: they carry the whole crypto beta.** Their Sharpes are **not directly
comparable** — a 1.4 Sharpe market-neutral book and a 1.0 Sharpe long-only book are different
animals with different risk.

So this notebook reports **two things**, never collapsing them:

- **Sharpe**, with the type (`market-neutral` / `directional`) stated on every row; and
- **the correlation of the CNN's long-short return stream with each benchmark's** — because the real
  questions are *"is the CNN just a repackaged reversal factor?"* (→ correlation with `STR`/`WSTR`)
  and *"is this supposedly-neutral book secretly long BTC?"* (→ correlation with buy-and-hold BTC).

A market-neutral book with a lower Sharpe than buy-and-hold BTC can still be worth having **if it is
uncorrelated** with BTC. A market-neutral book that is 0.9-correlated with `STR` is **not a CNN
result — it is short-term reversal wearing a CNN costume.** Both verdicts are stated plainly.

## Costs — the asymmetry is deliberate and favors the benchmarks

Long-short books are charged the locked grid (`2 × bps` per rebalance, both legs). Buy-and-hold
benchmarks are charged **nothing**, because holding does not trade. That asymmetry makes the
benchmarks *harder* to beat, which is the honest direction: the CNN should have to clear a
cost-free buy-and-hold bar while paying its own turnover.

## 0. Config — MUST match crypto_train.ipynb exactly (so rows align with the cached preds)

In [1]:
import os, time, json
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings('ignore')
from math import erf, sqrt

# ---- these MUST equal crypto_train.ipynb, or eval_base will not align with the .npy preds ----
IMAGE_SIZES = [5, 20]
HORIZONS    = [5, 20]
ANNUALIZER  = 365.0
COST_GRID   = [0, 10, 25, 50]
N_CONFIGS   = len(IMAGE_SIZES) * len(HORIZONS)
OOS_START, OOS_END = '2021-01-01', '2026-06-30'
TRAIN_END   = '2020-12-31'
ENSEMBLE_SIZE   = 5
DROP_INCOMPLETE = True

MA_LAGS   = [3, 5, 10, 20, 50, 100, 200]
MA_COLS   = [f'MA{L}' for L in MA_LAGS]
META_COLS = ['MOM','STR','WSTR'] + MA_COLS

PARQUET_PATH = './crypto_market_data_2018_2026.parquet'
IMG_DIR      = './crypto_image_store'
PRED_DIR     = './crypto_preds'
OUT_CSV      = './crypto_benchmark_results.csv'

def log(m): print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)

df_market = pd.read_parquet(PARQUET_PATH)
df_market['Date'] = pd.to_datetime(df_market['Date'])
log(f"parquet: {len(df_market):,} rows | {df_market['Symbol'].nunique()} assets | "
    f"{df_market['Date'].min().date()} -> {df_market['Date'].max().date()}")
log(f"benchmark families scored on the locked {N_CONFIGS}-config grid, ann={ANNUALIZER:.0f}, "
    f"costs {COST_GRID} bps/side")

[03:28:12] parquet: 887,326 rows | 662 assets | 2018-01-01 -> 2026-06-30
[03:28:12] benchmark families scored on the locked 4-config grid, ann=365, costs [0, 10, 25, 50] bps/side


## 1. Machinery copied verbatim from crypto_train.ipynb

These functions are **byte-for-byte the same** as in the training notebook. They must be, because
they reconstruct `eval_base` in the exact row order the cached `.npy` predictions were written in —
any divergence would silently misalign a prediction with the wrong (symbol, date). They are copied
rather than imported so this notebook stays standalone and cannot be broken by an edit to the other.

In [2]:
def _height(n_days): return 32 if n_days == 5 else (64 if n_days == 20 else 96)
def img_shape(n_days): return _height(n_days), n_days * 3

def store_paths(n_days):
    return (os.path.join(IMG_DIR, f'raw_n{n_days}.dat'),
            os.path.join(IMG_DIR, f'meta_n{n_days}.parquet'))

def align_store(m, df, verbose=False):
    pos_df = (df.sort_values(['Symbol', 'Date'])
                .assign(pos=lambda x: x.groupby('Symbol').cumcount(),
                        end_date=lambda x: x['Date'].dt.strftime('%Y-%m-%d'))
                [['Symbol', 'end_date', 'pos']]
                .rename(columns={'Symbol': 'symbol'}))
    mm = m.merge(pos_df, on=['symbol', 'end_date'], how='left')
    ok = mm['pos'].notna() & (mm['pos'].values == mm['end_idx'].values)
    return ok.values, mm

def load_store(n_days, df=None):
    dat, meta = store_paths(n_days)
    if not (os.path.exists(dat) and os.path.exists(meta)):
        return None
    m = pd.read_parquet(meta).reset_index(drop=True)
    H, W = img_shape(n_days); N = len(m)
    if os.path.getsize(dat) != N * H * W:
        raise RuntimeError(f"N={n_days}: .dat size mismatch vs metadata. Wrong dataset.")
    m['_row'] = np.arange(N)
    if DROP_INCOMPLETE:
        bad = m['end_idx'].values < (n_days - 1)
        if bad.any(): m = m[~bad].reset_index(drop=True)
    if df is not None:
        ok, _ = align_store(m, df)
        if (~ok).any(): m = m[ok].reset_index(drop=True)
        if len(m) == 0:
            raise RuntimeError(f"N={n_days}: NO rows survive alignment; re-render required.")
    return {'dat': dat, 'shape': (N, 1, H, W), 'meta': m}

def label_for_horizon(store, df, horizon):
    m = store['meta']
    closes, dates = {}, {}
    for s, g in df.groupby('Symbol'):
        g = g.sort_values('Date')
        closes[s] = g['Close'].values.astype(np.float64)
        dates[s]  = g['Date'].values.astype('datetime64[D]')
    fwd = np.full(len(m), np.nan)
    for s, pos in m.groupby('symbol').indices.items():
        c, dt = closes.get(s), dates.get(s)
        if c is None: continue
        ei = m['end_idx'].values[pos]; tgt = ei + horizon
        ok = tgt < len(c); tgt_c = np.minimum(tgt, len(c) - 1)
        span  = (dt[tgt_c] - dt[ei]).astype('timedelta64[D]').astype(int)
        exact = ok & (span == horizon)                 # EXACTLY horizon calendar days (24/7 fix)
        base_p, fwd_p = c[ei], c[tgt_c]
        with np.errstate(invalid='ignore', divide='ignore'):
            fwd[pos] = np.where(exact & (base_p != 0), fwd_p / base_p - 1.0, np.nan)
    d = m.copy(); d['Fwd_Ret'] = fwd
    d = d[~np.isnan(d['Fwd_Ret'].values)].reset_index(drop=True)
    rows  = d['_row'].values
    base  = d.rename(columns={'symbol': 'Symbol'})[['end_date','Symbol'] + META_COLS + ['Fwd_Ret']]
    base['Date'] = pd.to_datetime(base['end_date'])
    base = base.drop(columns='end_date')[['Date','Symbol'] + META_COLS + ['Fwd_Ret']]
    return rows, base

def test_positions(base):
    d = base['Date'].values
    te = np.where((d >= np.datetime64(OOS_START)) & (d <= np.datetime64(OOS_END)))[0]
    return te[np.argsort(d[te], kind='stable')]

def pred_path(n_days, horizon, seed):
    return os.path.join(PRED_DIR, f'pred_I{n_days}_R{horizon}_s{seed}.npy')

def rebalance_grid(dates, horizon):
    d0, d1 = pd.Timestamp(np.min(dates)), pd.Timestamp(np.max(dates))
    return pd.date_range(d0, d1, freq=f'{horizon}D')

def add_trend_factor(ev, horizon, n_beta=12, min_names=30):
    ev = ev.copy(); ev['TREND'] = np.nan
    rebal = rebalance_grid(ev['Date'].values, horizon)
    betas = {}
    for t in rebal:
        g = ev[ev['Date'] == t].dropna(subset=MA_COLS + ['Fwd_Ret'])
        if len(g) < min_names: continue
        X = np.column_stack([np.ones(len(g)), g[MA_COLS].values])
        b, *_ = np.linalg.lstsq(X, g['Fwd_Ret'].values, rcond=None); betas[t] = b
    known = []
    for t in rebal:
        if len(known) >= n_beta:
            bbar = np.mean(known[-n_beta:], axis=0)
            g = ev.loc[ev['Date'] == t, MA_COLS]; ok = g.notna().all(axis=1)
            if ok.any():
                X = np.column_stack([np.ones(int(ok.sum())), g[ok].values])
                ev.loc[g[ok].index, 'TREND'] = X @ bbar
        if t in betas: known.append(betas[t])
    return ev

def _ls_series(df, sig, H, min_names=20, cost_bps=0.0):
    d = df.dropna(subset=[sig, 'Fwd_Ret'])
    rebal = rebalance_grid(df['Date'].values, H)
    d = d[d['Date'].isin(rebal)]; out = {}
    for dt, g in d.groupby('Date'):
        if len(g) < min_names: continue
        dec = pd.qcut(g[sig].rank(method='first'), 10, labels=False)
        out[dt] = (g.loc[dec == 9, 'Fwd_Ret'].mean()
                   - g.loc[dec == 0, 'Fwd_Ret'].mean() - 2 * cost_bps / 1e4)
    return pd.Series(out).sort_index()

def _sharpe(r, H):
    r = r.dropna()
    return np.nan if len(r) < 2 or r.std(ddof=1) == 0 else \
           r.mean() / r.std(ddof=1) * np.sqrt(ANNUALIZER / H)

def _tstat(r):
    r = r.dropna()
    return np.nan if len(r) < 2 or r.std(ddof=1) == 0 else \
           r.mean() / (r.std(ddof=1) / np.sqrt(len(r)))

def _p_two_sided(t):
    return np.nan if np.isnan(t) else 2 * (1 - 0.5 * (1 + erf(abs(t) / sqrt(2))))

def _block_boot_ci(r, H, n_boot=2000, seed=0):
    r = r.dropna().values; n = len(r)
    if n < 10: return (np.nan, np.nan)
    avg_block = max(2, int(n ** (1/3))); p = 1.0 / avg_block
    rng = np.random.default_rng(seed); ann = np.sqrt(ANNUALIZER / H); est = np.empty(n_boot)
    for b in range(n_boot):
        idx = np.empty(n, dtype=int); i = 0
        while i < n:
            start = rng.integers(0, n); L = rng.geometric(p)
            for k in range(L):
                if i >= n: break
                idx[i] = (start + k) % n; i += 1
        s = r[idx]
        est[b] = s.mean() / s.std(ddof=1) * ann if s.std(ddof=1) > 0 else 0.0
    return (np.percentile(est, 2.5), np.percentile(est, 97.5))

def rebuild_eval_base(n_days, horizon):
    '''eval_base + CNN_Prob + TREND from CACHED predictions -- no training, no GPU.'''
    store = load_store(n_days, df_market)
    if store is None: return None
    rows, base = label_for_horizon(store, df_market, horizon)
    te = test_positions(base); eb = base.iloc[te].reset_index(drop=True)
    ps = [np.load(pred_path(n_days, horizon, 1000+i)) for i in range(ENSEMBLE_SIZE)
          if os.path.exists(pred_path(n_days, horizon, 1000+i))]
    if not ps or {len(p) for p in ps} != {len(te)}: return None
    eb['CNN_Prob'] = np.mean(ps, axis=0)
    return add_trend_factor(eb, horizon)

log("machinery loaded (verbatim copy of the crypto_train.ipynb definitions)")

[03:28:12] machinery loaded (verbatim copy of the crypto_train.ipynb definitions)


## 2. Kalman trend/vol state estimator — a latent trend the CNN must beat

Adapted directly from the pairs project's `models/kalman_filter.py::run_ecm_kalman`. That filter
tracks a **mean-reverting latent state** — it predicts `theta_pred = F·theta + c` with
`F = (1+λ)I` pulling the state back toward `theta_star`, then does a standard causal Kalman update.

Here the same machinery estimates a **local linear trend** for each coin instead of a cointegration
hedge ratio:

| | pairs `run_ecm_kalman` | this trend filter |
|---|---|---|
| state | `[β, α]` hedge ratio & intercept | `[level, slope]` |
| observation | `y = β·x + α` | `log(close) = level` |
| transition | `F = (1+λ)I` (state reverts to `theta_star`) | `A = [[1,1],[0,φ]]` (level integrates slope; **slope is damped toward 0 at rate φ** — the same mean-reversion idea, applied to the trend) |

`φ < 1` is the ECM's reversion, made concrete: trends **decay** rather than persist forever, which
is the honest prior for crypto. The filter is **strictly causal** — the state at `t` uses only
observations up to `t` — so there is no look-ahead (verified below with the same corruption test used
for `TREND` and the vol regime).

**Signal = filtered slope ÷ filtered vol** (an EWMA of the innovation size). Using both states — the
"trend/vol" estimator — makes it a *risk-adjusted* trend that is comparable across coins with wildly
different volatility, which a raw slope is not. Cross-sectionally rank on it, long-short the deciles.

The parameters (`φ`, the process/observation variances, the EWMA half-life) are **fixed and declared,
not tuned** — a cross-sectional *rank* is insensitive to their exact values, and every coin uses the
identical filter, so the comparison is fair.

In [3]:
# ---- fixed, pre-declared filter constants (NOT tuned; the cross-sectional rank is scale-robust) ----
KF_PHI      = 0.97      # slope damping  (the ECM (1+lambda) reversion, applied to the trend)
KF_R        = 2.5e-3    # observation noise var on log-price (~ (5%/day)^2)
KF_Q_LEVEL  = 2.5e-5    # level process var
KF_Q_SLOPE  = 4.0e-6    # slope process var (slope evolves slowly)
KF_EWMA_HL  = 20        # half-life (days) for the innovation-variance -> vol state
KF_WARMUP   = 30        # emit no signal until the filter has seen this many bars

def kf_trend_vol(logp):
    '''Causal local-linear-trend + vol filter. Returns (slope, vol) per bar; NaN during warm-up.
       Mirror of run_ecm_kalman: predict with a mean-reverting F, then standard Kalman update.'''
    n = len(logp)
    slope = np.full(n, np.nan); vol = np.full(n, np.nan)
    if n < KF_WARMUP + 2: return slope, vol
    A  = np.array([[1.0, 1.0], [0.0, KF_PHI]])          # level integrates slope; slope damped -> 0
    Q  = np.array([[KF_Q_LEVEL, 0.0], [0.0, KF_Q_SLOPE]])
    Hh = np.array([[1.0, 0.0]])                          # observe the level (log price)
    x  = np.array([logp[0], 0.0]); P = np.eye(2)
    lam = 1.0 - 0.5 ** (1.0 / KF_EWMA_HL); s2 = KF_R
    I2 = np.eye(2)
    for t in range(n):
        xp = A @ x                                       # predict (F @ theta)
        Pp = A @ P @ A.T + Q
        v  = logp[t] - (Hh @ xp).item()                  # innovation
        S  = (Hh @ Pp @ Hh.T).item() + KF_R
        if not np.isfinite(S) or S <= 0: break
        K  = (Pp @ Hh.T).flatten() / S                   # gain
        x  = xp + K * v                                  # update
        P  = (I2 - np.outer(K, Hh.flatten())) @ Pp
        s2 = (1.0 - lam) * s2 + lam * (v * v)            # EWMA innovation var -> vol state
        if t >= KF_WARMUP:
            slope[t] = x[1]; vol[t] = np.sqrt(s2)
    return slope, vol

# ---- compute the signal once per asset (it is horizon-independent: a property of the price path) ----
t0 = time.time()
kf_parts = []
for s, g in df_market.sort_values(['Symbol','Date']).groupby('Symbol'):
    lp = np.log(g['Close'].values.astype(np.float64))
    sl, vo = kf_trend_vol(lp)
    with np.errstate(invalid='ignore', divide='ignore'):
        sig = sl / vo                                    # risk-adjusted latent trend
    kf_parts.append(pd.DataFrame({'Symbol': s, 'Date': g['Date'].values, 'KALMAN': sig}))
kf_sig = pd.concat(kf_parts, ignore_index=True)
log(f"Kalman signal computed for {kf_sig['Symbol'].nunique()} assets in {time.time()-t0:.1f}s "
    f"({kf_sig['KALMAN'].notna().sum():,} non-NaN)")

# ---- leak audit: corrupt prices from t onward; the signal up to t-1 must be bit-identical ----
_s = df_market[df_market.Symbol == 'BTC'].sort_values('Date').reset_index(drop=True)
_lp = np.log(_s['Close'].values.astype(float)); i = 800
a_sl, _ = kf_trend_vol(_lp)
_lp2 = _lp.copy(); _lp2[i:] += 3.0                        # 20x price jump from t onward
b_sl, _ = kf_trend_vol(_lp2)
same = np.allclose(a_sl[:i][~np.isnan(a_sl[:i])], b_sl[:i][~np.isnan(b_sl[:i])])
log(f"Kalman leak audit: corrupting price from t={i} leaves every earlier slope identical -> "
    f"{'PASS' if same else 'FAIL'}")

[03:28:20] Kalman signal computed for 662 assets in 6.6s (867,553 non-NaN)
[03:28:20] Kalman leak audit: corrupting price from t=800 leaves every earlier slope identical -> PASS


## 3. Score every strategy on identical rows

For each config with cached predictions, rebuild the CNN's exact `eval_base`, merge in the Kalman
signal, and score all strategies on the same non-overlapping H-day rebalance grid.

- **Market-neutral (long-short D10−D1):** CNN, MOM, STR, WSTR, TREND, KALMAN — full cost grid.
- **Directional (long-only):** BTC buy-and-hold (BTC's own H-day forward return, rolled), and
  equal-weight top-100 (the cross-sectional mean forward return). Charged no cost — see the header.

In [4]:
MKT_NEUTRAL = ['CNN_Prob','MOM','STR','WSTR','TREND','KALMAN']

def directional_series(eval_base, horizon, which):
    '''Long-only benchmark return per rebalance date, from the CNN's own rows.'''
    rebal = rebalance_grid(eval_base['Date'].values, horizon)
    if which == 'BTC_BH':
        s = (eval_base[eval_base['Symbol'] == 'BTC'].set_index('Date')['Fwd_Ret'])
        return s[~s.index.duplicated()].reindex(rebal).dropna()
    if which == 'EW_TOP100':
        return eval_base.groupby('Date')['Fwd_Ret'].mean().reindex(rebal).dropna()
    raise ValueError(which)

CONFIGS = [(n, h) for n in IMAGE_SIZES for h in HORIZONS
           if os.path.exists(pred_path(n, h, 1000))]
log(f"configs with cached predictions: {[f'I{n}/R{h}' for n,h in CONFIGS]}")
missing = [f'I{n}/R{h}' for n in IMAGE_SIZES for h in HORIZONS
           if not os.path.exists(pred_path(n, h, 1000))]
if missing:
    log(f"NOT yet cached (skipped; re-run this cell after crypto_train finishes them): {missing}")

rows = []; SERIES = {}       # SERIES[(cfg)] -> {strategy: return series} for the correlation table
for n, h in CONFIGS:
    cfg = f"I{n}/R{h}"
    eb = rebuild_eval_base(n, h)
    if eb is None:
        log(f"{cfg}: preds missing/stale, skipping."); continue
    eb = eb.merge(kf_sig, on=['Symbol','Date'], how='left')
    SERIES[cfg] = {}

    for strat in MKT_NEUTRAL:
        r0 = _ls_series(eb, strat, h, cost_bps=0)
        SERIES[cfg][strat] = r0
        lo, hi = _block_boot_ci(r0, h)
        row = {'Config': cfg, 'Strategy': strat, 'Type': 'market-neutral',
               'N': int(r0.dropna().shape[0]), 'tstat': _tstat(r0),
               'boot_lo': lo, 'boot_hi': hi}
        for bps in COST_GRID:
            row[f'Sharpe_{bps}bps'] = _sharpe(_ls_series(eb, strat, h, cost_bps=bps), h)
        rows.append(row)

    for which in ['BTC_BH','EW_TOP100']:
        r0 = directional_series(eb, h, which)
        SERIES[cfg][which] = r0
        lo, hi = _block_boot_ci(r0, h)
        row = {'Config': cfg, 'Strategy': which, 'Type': 'directional',
               'N': int(r0.shape[0]), 'tstat': _tstat(r0), 'boot_lo': lo, 'boot_hi': hi}
        for bps in COST_GRID:                                  # buy-and-hold: no turnover -> no cost
            row[f'Sharpe_{bps}bps'] = _sharpe(r0, h)
        rows.append(row)
    log(f"{cfg}: scored {len(MKT_NEUTRAL)} market-neutral + 2 directional benchmarks")

bench = pd.DataFrame(rows)
scols = [f'Sharpe_{b}bps' for b in COST_GRID]

print("\n" + "="*94)
print("STAGE 3 MASTER TABLE  —  CNN Sharpe ALONGSIDE every benchmark (same rows, ann=365)")
print("="*94)
for cfg in [f"I{n}/R{h}" for n,h in CONFIGS]:
    sub = bench[bench.Config == cfg].copy()
    if sub.empty: continue
    order = MKT_NEUTRAL + ['BTC_BH','EW_TOP100']
    sub['_o'] = sub['Strategy'].map({s:i for i,s in enumerate(order)})
    sub = sub.sort_values('_o')
    print(f"\n{cfg}   (gross Sharpe, then net at the locked cost grid; 95% block-bootstrap CI on gross)")
    disp = sub[['Strategy','Type'] + scols + ['boot_lo','boot_hi','N']].copy()
    disp.columns = ['Strategy','Type'] + [f'{b}bps' for b in COST_GRID] + ['CI_lo','CI_hi','N']
    print(disp.to_string(index=False, float_format=lambda x: f'{x:+.3f}'))

[03:28:20] configs with cached predictions: ['I5/R5', 'I5/R20', 'I20/R5', 'I20/R20']
[03:28:27] I5/R5: scored 6 market-neutral + 2 directional benchmarks
[03:28:30] I5/R20: scored 6 market-neutral + 2 directional benchmarks
[03:28:38] I20/R5: scored 6 market-neutral + 2 directional benchmarks
[03:28:40] I20/R20: scored 6 market-neutral + 2 directional benchmarks

STAGE 3 MASTER TABLE  —  CNN Sharpe ALONGSIDE every benchmark (same rows, ann=365)

I5/R5   (gross Sharpe, then net at the locked cost grid; 95% block-bootstrap CI on gross)
 Strategy           Type   0bps  10bps  25bps  50bps  CI_lo  CI_hi   N
 CNN_Prob market-neutral +1.473 +1.228 +0.859 +0.245 +0.699 +2.333 401
      MOM market-neutral -0.161 -0.366 -0.672 -1.183 -0.947 +0.778 401
      STR market-neutral -0.291 -0.458 -0.708 -1.126 -0.973 +0.469 401
     WSTR market-neutral -0.765 -0.951 -1.230 -1.695 -1.586 +0.166 401
    TREND market-neutral +0.238 +0.019 -0.310 -0.857 -0.611 +1.045 389
   KALMAN market-neutral +0.706 +0

## 4. Is the CNN distinct — or a factor in disguise?

Sharpe rank alone cannot answer this. Two return streams can both have Sharpe ~1 and be **the same
trade**. So correlate the CNN's long-short return series with every benchmark's, on the shared
rebalance dates:

- high **`+`** correlation with `STR`/`WSTR` → the CNN *is* short-term reversal;
- high correlation with `BTC_BH` → the "market-neutral" book is quietly long the market;
- low correlation with everything **and** a competitive Sharpe → a genuinely new signal.

In [5]:
print("="*84)
print("CNN LONG-SHORT vs EACH BENCHMARK  —  correlation of the return streams")
print("="*84)
corr_rows = []
for cfg, sd in SERIES.items():
    cnn = sd['CNN_Prob']
    line = {'Config': cfg}
    for name, s in sd.items():
        if name == 'CNN_Prob': continue
        j = pd.concat([cnn, s], axis=1, join='inner').dropna()
        line[name] = (j.iloc[:,0].corr(j.iloc[:,1]) if len(j) > 5 else np.nan)
    corr_rows.append(line)
corr = pd.DataFrame(corr_rows).set_index('Config')
print(corr.to_string(float_format=lambda x: f'{x:+.2f}'))
print("\n  |corr| > ~0.7 with STR/WSTR  -> the CNN is largely that reversal factor.")
print("  |corr| > ~0.5 with BTC_BH    -> the book carries market beta; it is not truly neutral.")

# ============================================================================
print("\n" + "="*84)
print("VERDICT  —  does the CNN beat what you could do without it?")
print("="*84)
for cfg in [f"I{n}/R{h}" for n,h in CONFIGS]:
    sub = bench[bench.Config == cfg]
    if sub.empty: continue
    g = sub.set_index('Strategy')
    cnn_g  = g.loc['CNN_Prob','Sharpe_0bps']
    cnn_25 = g.loc['CNN_Prob','Sharpe_25bps']
    mn = [s for s in ['MOM','STR','WSTR','TREND','KALMAN'] if s in g.index]
    best_mn = g.loc[mn, 'Sharpe_0bps'].astype(float)
    best_name = best_mn.idxmax(); best_val = best_mn.max()
    btc = g.loc['BTC_BH','Sharpe_0bps'] if 'BTC_BH' in g.index else np.nan
    c = corr.loc[cfg] if cfg in corr.index else pd.Series(dtype=float)
    rev_corr = np.nanmax([abs(c.get('STR', np.nan)), abs(c.get('WSTR', np.nan))])
    btc_corr = abs(c.get('BTC_BH', np.nan))

    print(f"\n{cfg}:")
    print(f"  CNN gross {cnn_g:+.3f} | net@25bps {cnn_25:+.3f} | "
          f"best cross-sectional benchmark = {best_name} {best_val:+.3f} | buy&hold BTC {btc:+.3f}")
    beats = cnn_g > best_val
    print(f"  [{'YES' if beats else 'NO '}] CNN beats every cross-sectional factor gross"
          + ("" if beats else f"  <-- it does NOT beat {best_name}; not a result over known factors"))
    print(f"  [{'YES' if cnn_25 > 0 else 'NO '}] CNN still positive after 25 bps/side")
    if np.isfinite(rev_corr):
        print(f"  [{'OK ' if rev_corr < 0.7 else '!! '}] distinct from reversal "
              f"(max |corr| with STR/WSTR = {rev_corr:.2f}"
              + (")" if rev_corr < 0.7 else " -> largely a reversal factor)"))
    if np.isfinite(btc_corr):
        print(f"  [{'OK ' if btc_corr < 0.5 else '!! '}] market-neutral in practice "
              f"(|corr| with buy&hold BTC = {btc_corr:.2f})")

print("\nHONEST BOTTOM LINE: a config is a genuine CNN result only if it (a) beats every")
print("cross-sectional factor, (b) survives 25 bps, and (c) is not just reversal in a costume.")
print("Anything failing (a) is momentum/reversal you did not need a neural net for -- say so.")

bench.to_csv(OUT_CSV, index=False)
print(f"\nsaved -> {OUT_CSV}")

CNN LONG-SHORT vs EACH BENCHMARK  —  correlation of the return streams
          MOM   STR  WSTR  TREND  KALMAN  BTC_BH  EW_TOP100
Config                                                     
I5/R5   -0.01 -0.17 -0.22  +0.01   +0.13   -0.07      -0.08
I5/R20  -0.02 +0.20 -0.12  -0.02   +0.01   -0.04      -0.01
I20/R5  -0.08 -0.14 -0.17  +0.11   +0.18   +0.02      +0.08
I20/R20 -0.11 +0.05 -0.38  +0.10   +0.36   +0.03      +0.29

  |corr| > ~0.7 with STR/WSTR  -> the CNN is largely that reversal factor.
  |corr| > ~0.5 with BTC_BH    -> the book carries market beta; it is not truly neutral.

VERDICT  —  does the CNN beat what you could do without it?

I5/R5:
  CNN gross +1.473 | net@25bps +0.859 | best cross-sectional benchmark = KALMAN +0.706 | buy&hold BTC +0.513
  [YES] CNN beats every cross-sectional factor gross
  [YES] CNN still positive after 25 bps/side
  [OK ] distinct from reversal (max |corr| with STR/WSTR = 0.22)
  [OK ] market-neutral in practice (|corr| with buy&hold BTC = 